# Step 1 – Stratified 5-fold partitions

Labels patients **high** (BMI >= 30) or **low** (BMI < 25), drops the in-between patients, and splits the rest into 5 stratified folds for each seed. Each fold is saved as `<OUTPUT_DIR>/<group>_bmi_s<seed>_part<k>_datapack.csv` (`OUTPUT_DIR` is set in the first code cell).

Set the sample paths in the first code cell to your own data locations before running.

In [ ]:
import os

import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ---- User input ----
GROUPS = ["large_group"]      # cohort CSVs in DATA_DIR (without .csv)
SEEDS = [11, 35317]           # 11 is saved as s0011
N_SPLITS = 5
DATA_DIR = "/path/to/BMI_DATA"                  # folder containing large_group.csv  (SAMPLE PATH)
OUTPUT_DIR = "/path/to/BMI_DATA/datapack"       # partition CSVs are written here    (SAMPLE PATH)
HIGH_BMI = 30
LOW_BMI = 25
# --------------------

In [ ]:
def bmi_status(bmi):
    if bmi >= HIGH_BMI:
        return "high"
    if bmi < LOW_BMI:
        return "low"
    return "mid"


os.makedirs(OUTPUT_DIR, exist_ok=True)
partition_dict = {}

for group in GROUPS:
    table = pd.read_csv(os.path.join(DATA_DIR, f"{group}.csv"))
    compact = table[["block_name", "BMI"]].copy()
    compact["status"] = compact["BMI"].apply(bmi_status)

    # Mid-BMI patients are excluded; the original row number is kept in the 'index' column
    bmi_df = compact[compact["status"] != "mid"][["block_name", "status"]].reset_index()
    print(f"{group}: {len(bmi_df)} patients "
          f"(high={(bmi_df.status == 'high').sum()}, low={(bmi_df.status == 'low').sum()}), "
          f"{len(compact) - len(bmi_df)} mid-BMI patients dropped")

    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, random_state=seed, shuffle=True)
        folds = [bmi_df.iloc[test_idx] for _, test_idx in skf.split(bmi_df, bmi_df["status"])]
        partition_dict[(group, seed)] = folds

        for part, fold in enumerate(folds):
            print(f"  seed {seed} part {part}: high={(fold.status == 'high').sum()}, low={(fold.status == 'low').sum()}")

## Save the partitions

In [ ]:
for (group, seed), folds in partition_dict.items():
    for part, fold in enumerate(folds):
        out = os.path.join(OUTPUT_DIR, f"{group}_bmi_s{str(seed).zfill(4)}_part{part}_datapack.csv")
        fold.to_csv(out, index=False)
        print("saved", out)